In [17]:
import importlib
import sys
from pathlib import Path

import numpy as np
import pandas as pd

# Run from the research repository or its test/ folder.
_candidates = [Path.cwd(), *Path.cwd().parents,
               Path('/Users/kk985/PhD/Research/AllocationMultiplicity')]
ROOT = next((p for p in _candidates if (p / 'src/allocations.py').exists()), Path.cwd())
sys.path.insert(0, str(ROOT))
import src.allocations

importlib.reload(src.allocations)
from src.allocations import Allocations

In [ ]:
RASHOMON_EPS      = 0.01
ALLOCATION_EPS    = 0.1
N_MODELS          = 100
NEED_COL          = "gagne_sum_t"
EXP_NEED_COL      = "exp_need_{model}"
GROUP_COL         = "race"
METHOD            = "bootstrap"
MODEL_CLASS       = "nn_multiclass_finetune_full_76aad3c8aee4"
# MODEL_CLASS       = "nn_multiclass"
SAMPLE_SIZE       = 100
ITERS_PER_SEED    = 5
ROOT              = Path.cwd().parent

# beta=0 uses true needs; beta=1 reproduces the predicted-needs baseline.

In [19]:
fp_input = ROOT / "data/obermeyer/obermeyer_data_cleaned.csv"
data = pd.read_csv(fp_input)

In [20]:
for PRED_METHOD in ["average"]:
    for NEED_THRESHOLD in [1]:
        QUALIFICATION_COL = f"qualified_gagne_{NEED_THRESHOLD}"
        for SELECTION_RATE in [0.1]:
            for REPLACE_ALPHA in [0.0]:
                # print(f"pred method: {PRED_METHOD} | qualification: {QUALIFICATION_COL} | selection rate: {SELECTION_RATE} | replacement alpha: {REPLACE_ALPHA}")
                print(f"pred method: {PRED_METHOD} | qualification: {QUALIFICATION_COL} | selection rate: {SELECTION_RATE}")
                fp_preds          = ROOT / f"data/predictions/test_{METHOD}_{MODEL_CLASS}_{PRED_METHOD}_{QUALIFICATION_COL}.csv"
                fp_output_people  = ROOT / f"data/results/allocations/people_{METHOD}_{MODEL_CLASS}_{PRED_METHOD}_y-{QUALIFICATION_COL}_rate-{SELECTION_RATE}_size-{SAMPLE_SIZE}.csv"
                fp_output_opt = ROOT / f"data/results/allocations/opt_{METHOD}_{MODEL_CLASS}_{PRED_METHOD}_y-{QUALIFICATION_COL}_rate-{SELECTION_RATE}_size-{SAMPLE_SIZE}.csv"
                fp_output_lda_tru = ROOT / f"data/results/allocations/lda_tru_{METHOD}_{MODEL_CLASS}_{PRED_METHOD}_y-{QUALIFICATION_COL}_rate-{SELECTION_RATE}_size-{SAMPLE_SIZE}.csv"
                fp_output_lda_exp = ROOT / f"data/results/allocations/lda_exp_{METHOD}_{MODEL_CLASS}_{PRED_METHOD}_y-{QUALIFICATION_COL}_rate-{SELECTION_RATE}_size-{SAMPLE_SIZE}.csv"
                # fp_output_people  = ROOT / f"data/results/allocations/people_{METHOD}_{MODEL_CLASS}_{PRED_METHOD}_y-{QUALIFICATION_COL}_rate-{SELECTION_RATE}_size-{SAMPLE_SIZE}_alpha-{REPLACE_ALPHA}.csv"
                # fp_output_lda_exp = ROOT / f"data/results/allocations/lda_exp_{METHOD}_{MODEL_CLASS}_{PRED_METHOD}_y-{QUALIFICATION_COL}_rate-{SELECTION_RATE}_size-{SAMPLE_SIZE}_alpha-{REPLACE_ALPHA}.csv"

                df = pd.read_csv(fp_preds)
                df = df.merge(
                    data[["person_id", GROUP_COL]],
                    on="person_id", how="left", validate="many_to_one",
                )
                required_exp_need_cols = [Allocations.exp_need_col_for_model(f"m_{i}", EXP_NEED_COL) for i in range(1, N_MODELS + 1)]
                missing = [col for col in required_exp_need_cols if col not in df]
                if missing:
                    raise ValueError(f"Missing model-specific expected-need columns: {missing}")            

                a = Allocations(
                    df=df, sample_size=SAMPLE_SIZE, selection_rate=SELECTION_RATE,
                    qualification_col="y", need_col=NEED_COL, group_col=GROUP_COL,
                    rashomon_eps=RASHOMON_EPS, allocation_eps=ALLOCATION_EPS,
                    n_models=N_MODELS, iters_per_seed=ITERS_PER_SEED,
                )
                allocations_opt, allocations_lda_tru, allocations_lda_exp, people = a.get_allocations(
                    exp_need_col=EXP_NEED_COL, alpha=REPLACE_ALPHA, algorithms=["OPT", "LDA-tru", "LDA-exp"],
                )

                fp_output_opt.parent.mkdir(parents=True, exist_ok=True)
                fp_output_lda_tru.parent.mkdir(parents=True, exist_ok=True)
                fp_output_lda_exp.parent.mkdir(parents=True, exist_ok=True)
                fp_output_people.parent.mkdir(parents=True, exist_ok=True)

                allocations_opt.to_csv(fp_output_opt, index=False)
                allocations_lda_tru.to_csv(fp_output_lda_tru, index=False)
                allocations_lda_exp.to_csv(fp_output_lda_exp, index=False)
                people.to_csv(fp_output_people, index=False)

pred method: average | qualification: qualified_gagne_1 | selection rate: 0.1


[Seed 4]: 100%|██████████| 5/5 [01:13<00:00, 14.72s/it]
